In [1]:
# Imports
import pandas as pd
from openai import OpenAI
import os
import ast
import numpy as np
import json
import re

In [2]:
# Path to Excel file
file_path = r'C:\Users\andre.chan\OneDrive - OneWorkplace\Tribal VW\Ad Hoc\Language Analysis\ModelGPT Analysis\Data\modelGPT_outputs_id3_id7_troc_q3.xlsx'

all_sheets = pd.read_excel(file_path, sheet_name=None)

In [3]:
modelgpt_df = pd.DataFrame(columns=[
    'id',
    'timestamp',
    'name',
    'userId',
    'sessionId',
    'tags',
    'input',
    'output'
])

# Loop through each sheet
for sheet_name, sheet_df in all_sheets.items():

    # Keep only columns that exist in both dataframes
    matching_cols = [col for col in modelgpt_df.columns if col in sheet_df.columns]

    # Select matching columns
    temp_df = sheet_df[matching_cols]

    # Append to main dataframe
    modelgpt_df = pd.concat([modelgpt_df, temp_df], ignore_index=True)

# Apply to all string columns and clean up unreadable strings
for col in modelgpt_df.select_dtypes(include='object').columns:
    modelgpt_df[col] = (
        modelgpt_df[col]
        .astype(str)
        .str.replace('‚Äô', "'", regex=False)
        .str.replace('â€™', "'", regex=False)
        .str.replace('‚Äú', '"', regex=False)
        .str.replace('‚Äù', '"', regex=False)
        .str.replace('â€œ', '"', regex=False)
        .str.replace('â€', '"', regex=False)
        .str.replace('â€“', '-', regex=False)
    )

modelgpt_df = modelgpt_df.replace('nan', np.nan)
modelgpt_df = modelgpt_df.dropna(how='all')
modelgpt_df = modelgpt_df.reset_index(drop=True)
modelgpt_df.head()


C:\Users\andre.chan\AppData\Local\Temp\ipykernel_26276\1204363620.py:38: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  modelgpt_df = modelgpt_df.replace('nan', np.nan)


,id,timestamp,name,userId,sessionId,tags,input,output
0,4781019c-c929-4a1b-a719-3239a648b527,2026-05-31T23:17:28.519Z,Chat Conversation for model 30702,3b6428ce-5e6b-40de-8a5f-b1d5054ca13b,NaN,"[""other""]","{\text\"":\""What colours are available?\""}""","""{\""message\"":\""The T-Roc is available in a va..."
1,05cd57ff-e9cb-47e6-82ba-c27882f5b569,2026-07-01T17:35:28.495Z,Chat Conversation for model 30702,b7120856-5569-4aaa-ab02-bf943c7d2613,NaN,"[""chat-model: 30702"",""color""]","{\text\"":\""Green\""}""","""{\""message\"":[{\""model\"":\""30702\"",\""trim\"":\..."
2,0708a3a7-3678-46b6-bc18-1290fd4707d9,2026-07-01T13:09:50.807Z,Chat Conversation for model 30275,235842df-ebed-401d-851b-349a9ff3db28,NaN,"[""car_range"",""chat-model: 30275""]","{\text\"":\""What's the ID.3 range?\""}""","""{\""message\"":\""```json\\n[\\n {\\n ..."
3,074bd0a6-3cc9-4c54-b344-ef7ad1b9af7c,2026-07-01T19:21:55.252Z,Chat Conversation for model 30702,4df9ab21-95c9-4846-84f4-af6d9567419b,NaN,"[""chat-model: 30702"",""color""]","{\text\"":\""Show me the colours \""}""","""{\""message\"":[{\""model\"":\""30702\"",\""trim\"":\..."
4,07744113-5e36-45c3-b860-890ff9b48f38,2026-07-01T19:38:08.022Z,Chat Conversation for model 30702,b970ba93-7838-44cf-b5b9-673b306244fc,NaN,"[""other""]","{\text\"":\""Which versions of the T-Roc are ava...","""{\""message\"":\""The new T-Roc is a new-generat..."


In [4]:
# Clean JSON objects in 'input' and 'output' columns
modelgpt_df['input'] = (
    modelgpt_df['input']
    .str.extract(r'\\":\\"(.*?)\\"', expand=False)
)

def extract_message(output_str):
    if not isinstance(output_str, str):
        return None
    
    try:
        outer = json.loads(output_str)
        if isinstance(outer, str):
            outer = json.loads(outer)
        
        message_content = outer["message"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return None

    try:
        if isinstance(message_content, list):
            return message_content[0]["message"]
        if isinstance(message_content, dict):
            return message_content["message"]
        
        clean = re.sub(r'```json\s*|\s*```', '', message_content).strip()
        inner = json.loads(clean)
        
        if isinstance(inner, list):
            return inner[0]["message"]
        return inner["message"]
    
    except (json.JSONDecodeError, KeyError):
        return message_content

modelgpt_df['output'] = modelgpt_df['output'].apply(extract_message)

# modelgpt_df['timestamp'] = pd.to_datetime(modelgpt_df['timestamp']).dt.date
# modelgpt_df = modelgpt_df.rename(columns={'timestamp': 'date'})

modelgpt_df["timestamp"] = pd.to_datetime(modelgpt_df["timestamp"])
modelgpt_df["timestamp"] = modelgpt_df["timestamp"].dt.tz_localize(None)
modelgpt_df['model'] = np.nan

#Label models
modelgpt_df.loc[
    modelgpt_df['name'].str.contains('30550', na=False),
    'model'
] = 'ID.7'

modelgpt_df.loc[
    modelgpt_df['name'].str.contains('30275', na=False),
    'model'
] = 'ID.3'

modelgpt_df.loc[
    modelgpt_df['name'].str.contains('30702', na=False),
    'model'
] = 'T-Roc'

modelgpt_df.loc[
    modelgpt_df['name'].str.contains('72014', na=False),
    'model'
] = 'California'


modelgpt_df

C:\Users\andre.chan\AppData\Local\Temp\ipykernel_26276\4173153047.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ID.7' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  modelgpt_df.loc[


,id,timestamp,name,userId,sessionId,tags,input,output,model
0,4781019c-c929-4a1b-a719-3239a648b527,2026-05-31 23:17:28.519,Chat Conversation for model 30702,3b6428ce-5e6b-40de-8a5f-b1d5054ca13b,NaN,"[""other""]",What colours are available?,The T-Roc is available in a variety of attract...,T-Roc
1,05cd57ff-e9cb-47e6-82ba-c27882f5b569,2026-07-01 17:35:28.495,Chat Conversation for model 30702,b7120856-5569-4aaa-ab02-bf943c7d2613,NaN,"[""chat-model: 30702"",""color""]",Green,User is asking about car colours,T-Roc
2,0708a3a7-3678-46b6-bc18-1290fd4707d9,2026-07-01 13:09:50.807,Chat Conversation for model 30275,235842df-ebed-401d-851b-349a9ff3db28,NaN,"[""car_range"",""chat-model: 30275""]",What's the ID.3 range?,I can help you with the range for the ID.3! Th...,ID.3
3,074bd0a6-3cc9-4c54-b344-ef7ad1b9af7c,2026-07-01 19:21:55.252,Chat Conversation for model 30702,4df9ab21-95c9-4846-84f4-af6d9567419b,NaN,"[""chat-model: 30702"",""color""]",Show me the colours,User is asking about car colours,T-Roc
4,07744113-5e36-45c3-b860-890ff9b48f38,2026-07-01 19:38:08.022,Chat Conversation for model 30702,b970ba93-7838-44cf-b5b9-673b306244fc,NaN,"[""other""]",Which versions of the T-Roc are available?,The new T-Roc is a new-generation compact SUV ...,T-Roc
...,...,...,...,...,...,...,...,...,...
17917,c4b51367-57e9-489d-b7ae-e73e08533b70,2026-08-25 06:46:14.446,Chat Conversation for model 30702,8183ee15-3873-4c6f-9e7f-4e4ff03875b5,NaN,"[""other""]",Which versions of the T-Roc are available?,The new T-Roc is a new-generation compact SUV ...,T-Roc
17918,c544e3bc-bede-416d-b3e0-e561ed410004,2026-08-25 06:46:49.078,Chat Conversation for model 30702,8183ee15-3873-4c6f-9e7f-4e4ff03875b5,NaN,"[""chat-model: 30702"",""other""]",what versions of t-Roc are available as 4x4 or...,I'm afraid none of the T-Roc models currently ...,T-Roc
17919,d01692de-efac-41c6-9779-57216540600c,2026-08-25 06:55:26.944,Chat Conversation for model 30702,c07599f7-889a-4ca1-8ceb-cd3e33904047,NaN,"[""chat-model: 30702"",""other""]",Tell me about the new T-Roc?,The new T-Roc is our stylish and versatile SUV...,T-Roc
17920,d40969ad-96a1-45fb-b395-b45bf8f5c0ac,2026-08-25 09:11:52.685,Chat Conversation for model 30702,de553cd4-32ec-4a56-a4d6-19cc8fb79f86,NaN,"[""chat-model: 30702"",""other""]","t,-roc r line",The **T-Roc R-Line** is the top trim in the T-...,T-Roc


In [5]:
modelgpt_df.to_excel('modelgpt_output_id3_id7_troc_q3.xlsx', index=False)

In [6]:
conversation_length = (
    modelgpt_df.groupby(["userId", "model"])["timestamp"]
      .agg(
          start_time="min",
          end_time="max"
      )
      .reset_index()
)

conversation_length["chat_duration"] = (
    conversation_length["end_time"]
    - conversation_length["start_time"]
)

datetime_cols = ["chat_duration", "start_time", "end_time"]

for col in conversation_length.select_dtypes(include=["datetimetz"]).columns:
    conversation_length[col] = conversation_length[col].dt.tz_localize(None)

conversation_length

,userId,model,start_time,end_time,chat_duration
0,00015c98-0ec1-4340-8f71-645f611a9ef3,T-Roc,2026-07-26 18:04:58.329,2026-07-26 18:04:58.329,0 days 00:00:00
1,00055ee7-54c3-4553-9d8e-ac5d75915af2,T-Roc,2026-07-27 20:52:08.907,2026-07-27 20:52:08.907,0 days 00:00:00
2,000b2964-fec0-41ef-b2e2-dba4dc03edd7,T-Roc,2026-08-31 08:57:02.378,2026-08-31 08:57:02.378,0 days 00:00:00
3,000db3ed-cfbd-4107-875f-cb02f1a3d93e,T-Roc,2026-07-27 03:55:07.156,2026-07-27 03:55:07.156,0 days 00:00:00
4,0013f487-0c4f-4cb1-9db2-4b76c174988f,T-Roc,2026-08-09 11:39:10.441,2026-08-09 11:39:10.441,0 days 00:00:00
...,...,...,...,...,...
13885,ffea7c09-86c1-49cb-ae91-299423f97880,ID.3,2026-08-10 11:12:15.276,2026-08-10 11:12:30.799,0 days 00:00:15.523000
13886,ffeab9c4-14df-46d2-bc9c-663fc777b668,ID.7,2026-08-19 14:24:23.128,2026-08-19 14:24:23.128,0 days 00:00:00
13887,fff61bb6-48a6-4958-8319-2cd0689edb85,T-Roc,2026-06-25 21:47:21.096,2026-06-25 21:47:21.096,0 days 00:00:00
13888,fffa6e1d-f670-45b1-92b1-57e6984c1448,ID.3,2026-06-12 19:28:22.833,2026-06-12 19:28:22.833,0 days 00:00:00


In [7]:
conversation_length.to_excel('chat_duration_q3.xlsx', index=False)

In [ ]:
# Read the combined dataset
df_combined = pd.read_excel(r"C:\Users\andre.chan\OneDrive - OneWorkplace\Tribal VW\Ad Hoc\Language Analysis\ModelGPT Analysis\data\modelgpt_2026_combined.xlsx")
df_combined["timestamp"] = pd.to_datetime(df_combined["timestamp"])
df_combined.head()

In [ ]:
# conversation_length = (
#     df_combined.groupby(["userId", "model"])["timestamp"]
#       .agg(
#           start_time="min",
#           end_time="max"
#       )
#       .reset_index()
# )

# conversation_length["chat_duration"] = (
#     conversation_length["end_time"]
#     - conversation_length["start_time"]
# )

# datetime_cols = ["chat_duration", "start_time", "end_time"]

# for col in conversation_length.select_dtypes(include=["datetimetz"]).columns:
#     conversation_length[col] = conversation_length[col].dt.tz_localize(None)

# conversation_length

In [ ]:
# conversation_length.to_excel('chat_duration.xlsx', index=False)

file_path = "modelgpt_output_conversationid_breakdown_v2.xlsx"

mode = "a" if os.path.exists(file_path) else "w"

with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode=mode,
    if_sheet_exists="replace" if mode == "a" else None
) as writer:
    conversation_length.to_excel(
        writer,
        sheet_name="chat_duration",
        index=False
    )